# Whoop Fitness Dataset - Quick Start Guide

Welcome! This notebook provides a quick introduction to the Whoop Fitness Dataset.

## What You'll Learn:
1. Load and explore the data
2. Understand key metrics
3. Create visualizations
4. Build a simple prediction model

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✅ Libraries loaded successfully!')

## 1. Load the Data

In [ ]:
# Load dataset
df = pd.read_csv('whoop_fitness_dataset_100k.csv')

# Convert date column
df['date'] = pd.to_datetime(df['date'])

print(f"Dataset Shape: {df.shape}")
print(f"Columns: {df.shape[1]}")
print(f"Rows: {df.shape[0]:,}")
print(f"\nDate Range: {df['date'].min()} to {df['date'].max()}")
print(f"Unique Users: {df['user_id'].nunique():,}")

In [ ]:
# First look at the data
df.head(10)

In [ ]:
# Dataset info
df.info()

In [ ]:
# Summary statistics
df.describe()

## 2. Understand the Data

In [ ]:
# Key metrics overview
print("KEY METRICS OVERVIEW")
print("="*60)
print(f"Average Recovery Score: {df['recovery_score'].mean():.1f}%")
print(f"Average Strain: {df['day_strain'].mean():.2f}")
print(f"Average Sleep: {df['sleep_hours'].mean():.2f} hours")
print(f"Average HRV: {df['hrv'].mean():.1f} ms")
print(f"Average Resting HR: {df['resting_heart_rate'].mean():.1f} bpm")
print(f"\nWorkout Completion Rate: {df['workout_completed'].mean()*100:.1f}%")

In [ ]:
# User demographics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gender distribution
df['gender'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('Gender Distribution', fontweight='bold', fontsize=14)
axes[0].set_ylabel('Count')

# Fitness level
fitness_order = ['Beginner', 'Intermediate', 'Advanced', 'Elite']
df['fitness_level'].value_counts().reindex(fitness_order).plot(kind='bar', ax=axes[1], color='mediumseagreen')
axes[1].set_title('Fitness Level Distribution', fontweight='bold', fontsize=14)
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

# Age distribution
df['age'].hist(bins=20, ax=axes[2], color='purple', edgecolor='black')
axes[2].set_title('Age Distribution', fontweight='bold', fontsize=14)
axes[2].set_xlabel('Age')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 3. Visualize Key Metrics

In [ ]:
# Recovery, Strain, Sleep, and HRV distributions
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].hist(df['recovery_score'], bins=40, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Recovery Score (%)', fontweight='bold')
axes[0, 0].set_ylabel('Frequency', fontweight='bold')
axes[0, 0].set_title('Recovery Score Distribution', fontsize=14)
axes[0, 0].axvline(df['recovery_score'].mean(), color='red', linestyle='--', linewidth=2,
                  label=f"Mean: {df['recovery_score'].mean():.1f}%")
axes[0, 0].legend()

axes[0, 1].hist(df['day_strain'], bins=40, color='orangered', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Day Strain', fontweight='bold')
axes[0, 1].set_ylabel('Frequency', fontweight='bold')
axes[0, 1].set_title('Strain Distribution', fontsize=14)
axes[0, 1].axvline(df['day_strain'].mean(), color='red', linestyle='--', linewidth=2,
                  label=f"Mean: {df['day_strain'].mean():.2f}")
axes[0, 1].legend()

axes[1, 0].hist(df['sleep_hours'], bins=40, color='mediumseagreen', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Sleep Hours', fontweight='bold')
axes[1, 0].set_ylabel('Frequency', fontweight='bold')
axes[1, 0].set_title('Sleep Duration Distribution', fontsize=14)
axes[1, 0].axvline(df['sleep_hours'].mean(), color='red', linestyle='--', linewidth=2,
                  label=f"Mean: {df['sleep_hours'].mean():.2f} hrs")
axes[1, 0].legend()

axes[1, 1].hist(df['hrv'], bins=40, color='purple', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('HRV (ms)', fontweight='bold')
axes[1, 1].set_ylabel('Frequency', fontweight='bold')
axes[1, 1].set_title('Heart Rate Variability Distribution', fontsize=14)
axes[1, 1].axvline(df['hrv'].mean(), color='red', linestyle='--', linewidth=2,
                  label=f"Mean: {df['hrv'].mean():.1f} ms")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Activity analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Activity types (when workout completed)
workout_data = df[df['workout_completed'] == 1]
activity_counts = workout_data['activity_type'].value_counts()
axes[0].barh(activity_counts.index, activity_counts.values, color='coral', edgecolor='black')
axes[0].set_xlabel('Number of Workouts', fontweight='bold')
axes[0].set_title('Workout Distribution by Activity Type', fontsize=14, fontweight='bold')

# Strain by activity
strain_by_activity = workout_data.groupby('activity_type')['activity_strain'].mean().sort_values(ascending=True)
axes[1].barh(strain_by_activity.index, strain_by_activity.values, color='steelblue', edgecolor='black')
axes[1].set_xlabel('Average Strain', fontweight='bold')
axes[1].set_title('Average Strain by Activity Type', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Explore Relationships

In [ ]:
# Key correlations
correlation_features = ['recovery_score', 'day_strain', 'sleep_hours', 'sleep_efficiency',
                       'hrv', 'resting_heart_rate', 'sleep_performance']

corr_matrix = df[correlation_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
           square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - Key Metrics', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: Sleep vs Recovery and HRV vs Recovery
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sleep vs Recovery
axes[0].scatter(df['sleep_hours'], df['recovery_score'], alpha=0.1, color='mediumseagreen')
axes[0].set_xlabel('Sleep Hours', fontweight='bold')
axes[0].set_ylabel('Recovery Score (%)', fontweight='bold')
axes[0].set_title('Sleep Hours vs Recovery Score', fontsize=14, fontweight='bold')
corr_sleep = df['sleep_hours'].corr(df['recovery_score'])
axes[0].text(0.05, 0.95, f'Correlation: {corr_sleep:.3f}',
            transform=axes[0].transAxes, bbox=dict(boxstyle='round', facecolor='wheat'),
            verticalalignment='top', fontweight='bold')

# HRV vs Recovery
axes[1].scatter(df['hrv'], df['recovery_score'], alpha=0.1, color='purple')
axes[1].set_xlabel('HRV (ms)', fontweight='bold')
axes[1].set_ylabel('Recovery Score (%)', fontweight='bold')
axes[1].set_title('HRV vs Recovery Score', fontsize=14, fontweight='bold')
corr_hrv = df['hrv'].corr(df['recovery_score'])
axes[1].text(0.05, 0.95, f'Correlation: {corr_hrv:.3f}',
            transform=axes[1].transAxes, bbox=dict(boxstyle='round', facecolor='wheat'),
            verticalalignment='top', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Build a Simple Prediction Model

Let's predict next-day recovery score!

In [ ]:
# Feature engineering: Create next-day recovery target
df_sorted = df.sort_values(['user_id', 'date']).reset_index(drop=True)
df_sorted['next_day_recovery'] = df_sorted.groupby('user_id')['recovery_score'].shift(-1)

# Select features
features = ['day_strain', 'sleep_hours', 'sleep_efficiency', 'hrv', 
           'resting_heart_rate', 'age', 'workout_completed', 'deep_sleep_hours']

# Prepare data
model_data = df_sorted[features + ['next_day_recovery']].dropna()

X = model_data[features]
y = model_data['next_day_recovery']

print(f"Training dataset: {X.shape[0]:,} samples")
print(f"Features: {len(features)}")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Test set: {X_test.shape[0]:,} samples")

In [ ]:
# Train Random Forest model
print("Training Random Forest model...")
rf_model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
print("✅ Model trained!")

In [ ]:
# Make predictions
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

# Evaluate
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
train_r2 = r2_score(y_train, y_pred_train)
test_r2 = r2_score(y_test, y_pred_test)

print("MODEL PERFORMANCE")
print("="*60)
print(f"Train RMSE: {train_rmse:.3f}")
print(f"Test RMSE:  {test_rmse:.3f}")
print(f"Train R²:   {train_r2:.3f}")
print(f"Test R²:    {test_r2:.3f}")
print("\nInterpretation:")
print(f"The model can predict next-day recovery within ±{test_rmse:.1f}% on average")
print(f"It explains {test_r2*100:.1f}% of the variance in recovery scores")

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Predictions vs Actual
axes[0].scatter(y_test, y_pred_test, alpha=0.3, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
            'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Recovery Score', fontweight='bold')
axes[0].set_ylabel('Predicted Recovery Score', fontweight='bold')
axes[0].set_title('Model Predictions vs Actual', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Prediction errors
errors = y_test.values - y_pred_test
axes[1].hist(errors, bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Prediction Error', fontweight='bold')
axes[1].set_ylabel('Frequency', fontweight='bold')
axes[1].set_title('Error Distribution', fontsize=14, fontweight='bold')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'],
        color='teal', edgecolor='black', alpha=0.8)
plt.xlabel('Importance', fontweight='bold', fontsize=12)
plt.ylabel('Feature', fontweight='bold', fontsize=12)
plt.title('Feature Importance for Recovery Prediction', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nMost Important Features:")
print(feature_importance.sort_values('importance', ascending=False))

## 6. Next Steps

Now that you've explored the basics, here are some ideas:

### Deeper Analysis
- Analyze weekly patterns (weekday vs weekend)
- Study recovery zones (Green/Yellow/Red)
- Examine sleep architecture impact
- Compare fitness levels

### Advanced Models
- Add lag features (previous 7 days)
- Try XGBoost or LightGBM
- Build user-specific models
- Create workout recommendation system

### Visualizations
- Time series trends for individual users
- Heatmaps of recovery by day of week
- Heart rate zone distributions
- Interactive dashboards

### Machine Learning
- Classification: Predict workout intensity
- Clustering: Segment users by patterns
- Anomaly detection: Find unusual health patterns
- Time series forecasting: 7-day recovery predictions

---

**Happy analyzing! 🚀**